# Lab 6 Part 2: NanoGPT - Transformer Language Model

## 🎯 Learning Objectives

By the end of this lab, you will:
- Understand why transformers are better than MLPs for sequences
- Understand attention mechanism (Query, Key, Value)
- Build a mini GPT using transformer architecture
- Implement different sampling strategies (temperature, top-k)
- Compare transformer vs makemore performance

**What You'll Build:** A Shakespeare text generator using transformers

**Architecture:** 4-layer transformer with 4 attention heads

**Reference:** [Karpathy's nanoGPT](https://github.com/karpathy/nanoGPT)

**Prerequisites:** Complete Lab 6 Part 1 (Makemore) first!

## Part 0: Setup

Same as makemore - check GPU, download data, create tokenizer

In [ ]:
import sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
import matplotlib.pyplot as plt

# Set random seeds
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Detect device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("\n✓ GPU detected! Training will take ~8 minutes.")
else:
    print("\n⚠️  No GPU detected. Training will take much longer (~60 min).")
    print("In Colab: Runtime → Change runtime type → Hardware accelerator → T4 GPU")

In [ ]:
# Download dataset if in Colab
import os

if 'google.colab' in sys.modules:
    if not os.path.exists('shakespeare.txt'):
        print("Downloading shakespeare.txt...")
        !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O shakespeare.txt
        print("✓ Download complete!")
    else:
        print("✓ shakespeare.txt already exists")
else:
    print("Running locally - ensure shakespeare.txt is in current directory")

# Load text
with open('shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"\nDataset size: {len(text):,} characters")

In [ ]:
# Create tokenizer (same as makemore)
class CharTokenizer:
    """Simple character-level tokenizer."""
    
    def __init__(self, text):
        chars = sorted(list(set(text)))
        self.vocab_size = len(chars)
        self.char2id = {ch: i for i, ch in enumerate(chars)}
        self.id2char = {i: ch for i, ch in enumerate(chars)}
    
    def encode(self, text):
        return [self.char2id[ch] for ch in text]
    
    def decode(self, ids):
        return ''.join([self.id2char[i] for i in ids])

tokenizer = CharTokenizer(text)
print(f"Vocabulary size: {tokenizer.vocab_size}")

# Prepare data
data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Train size: {len(train_data):,} tokens")
print(f"Val size: {len(val_data):,} tokens")
print("\n✓ Setup complete!")

In [ ]:
# Data loading function (same as makemore)
def get_batch(split, train_data, val_data, batch_size=32, block_size=8, device='cpu'):
    """Generate a batch of (context, target) pairs."""
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

## Part 1: Why Transformers?

### Limitations of Makemore

**Problem 1: Fixed Context Window**
- Makemore uses only 8 characters of context
- Can't remember earlier parts of sentence
- Can't scale to longer context (16, 32, 64+ tokens)

**Problem 2: MLP Treats All Tokens Equally**
- Flattens all embeddings together
- Can't focus on relevant parts
- Can't learn which tokens matter most for prediction

**Example:**
```
Context: "The cat sat on the mat. The dog ran to the"
Predicting: "park" or "tree"

Makemore (8 char context): Only sees "to the" - not enough!
Transformer (64+ token context): Sees full sentence, attends to "dog" and "ran"
```

### What Transformers Solve

**1. Attention Mechanism:** Look at ALL previous tokens, focus on relevant ones

**2. Positional Encoding:** Know where tokens are in sequence

**3. Scalable Architecture:** Can increase context length without changing architecture

**4. Parallelizable:** All tokens processed simultaneously (unlike RNNs)

### Results

- Makemore: Perplexity ~7.5 (8-char context)
- NanoGPT: Perplexity ~5.0 (64-char context)
- More coherent text, better long-range dependencies

## Part 2: Attention Mechanism

### What is Attention?

**Attention = weighted average of past tokens**

Instead of treating all context equally, the model learns to:
- Focus on relevant tokens
- Ignore irrelevant tokens
- Combine information from multiple tokens

### Query, Key, Value Intuition

**Analogy: Library Search**

- **Query (Q):** "I'm looking for books about machine learning"
- **Key (K):** Book tags/descriptions ("ML", "AI", "Deep Learning")
- **Value (V):** Actual book content

**Process:**
1. Compare Query with each Key (compute similarity)
2. Higher similarity = higher attention weight
3. Weighted sum of Values = output

### Mathematical Formulation

```
Attention(Q, K, V) = softmax(Q·K^T / √d_k) · V
```

**Steps:**
1. **Compute scores:** Q · K^T (dot product)
   - High dot product = query matches key = relevant token
2. **Scale:** Divide by √d_k (stabilize gradients)
3. **Normalize:** Softmax → attention weights (sum to 1)
4. **Weighted sum:** Multiply by V, sum up

### Self-Attention vs Cross-Attention

**Self-Attention:** Attend to tokens in the same sequence
- Q, K, V all from same input
- Used in GPT (decoder-only)
- "What parts of my input should I focus on?"

**Cross-Attention:** Attend to tokens in a different sequence
- K, V from source, Q from target
- Used in encoder-decoder models (translation)
- "What parts of the source should I focus on for this target word?"

**We'll use self-attention (GPT-style)!**

### Interactive Attention Visualization

See how attention weights are computed and applied!

In [ ]:
%%html
<style>
#attn-container * { box-sizing: border-box; margin: 0; padding: 0; }
#attn-container { padding: 1rem 0; font-family: sans-serif; color: #1a1a1a; }
#attn-container h1 { font-size: 18px; font-weight: 500; margin-bottom: 4px; }
#attn-container .subtitle { font-size: 13px; color: #666; margin-bottom: 20px; }
#attn-container .section { margin-bottom: 16px; }
#attn-container .section-label { font-size: 11px; font-weight: 500; color: #666; text-transform: uppercase; letter-spacing: .04em; margin-bottom: 8px; }
#attn-container .tokens { display: flex; gap: 4px; margin-bottom: 12px; }
#attn-container .token { padding: 8px 12px; border-radius: 6px; background: #f7f7f5; border: 1px solid #ddd; font-size: 13px; cursor: pointer; transition: all 0.2s; }
#attn-container .token:hover { border-color: #185FA5; }
#attn-container .token.selected { background: #E6F1FB; border-color: #185FA5; font-weight: 500; }
#attn-container .attn-matrix { display: grid; gap: 2px; margin-top: 12px; }
#attn-container .attn-cell { height: 40px; border-radius: 4px; display: flex; align-items: center; justify-content: center; font-size: 11px; color: #fff; font-weight: 500; transition: all 0.2s; }
#attn-container .info-box { padding: 12px; border-radius: 8px; background: #f7f7f5; border: 0.5px solid #ddd; font-size: 12px; line-height: 1.6; }
#attn-container .formula { font-family: monospace; background: #fff; padding: 8px; border-radius: 6px; border: 0.5px solid #ddd; margin: 8px 0; }
</style>

<div id="attn-container">
  <h1>Attention Mechanism Playground</h1>
  <p class="subtitle">Click a token to see where it attends (causal masking: can only attend to past tokens).</p>

  <div class="section">
    <div class="section-label">Input Sequence</div>
    <div class="tokens" id="attn-tokens"></div>
  </div>

  <div class="section">
    <div class="section-label">Attention Weights (lighter = higher attention)</div>
    <div class="attn-matrix" id="attn-matrix"></div>
  </div>

  <div class="info-box" id="attn-info">
    Click a token above to see its attention weights.
  </div>
</div>

<script>
(function() {
  const TOKENS = ['The', 'cat', 'sat', 'on', 'the', 'mat'];
  
  const ATTN = [
    [1.0, 0, 0, 0, 0, 0],
    [0.3, 0.7, 0, 0, 0, 0],
    [0.1, 0.5, 0.4, 0, 0, 0],
    [0.1, 0.1, 0.2, 0.6, 0, 0],
    [0.2, 0.1, 0.1, 0.4, 0.2, 0],
    [0.05, 0.05, 0.1, 0.3, 0.1, 0.4]
  ];
  
  let selected = 0;
  
  function getColor(weight) {
    if (weight === 0) return '#f0f0f0';
    const intensity = Math.floor((1 - weight) * 200 + 55);
    return `rgb(${intensity}, ${intensity + 20}, ${255})`;
  }
  
  function renderTokens() {
    const tokensEl = document.getElementById('attn-tokens');
    if (!tokensEl) return;
    tokensEl.innerHTML = TOKENS.map((t, i) => 
      `<div class="token ${i === selected ? 'selected' : ''}" onclick="attnSelect(${i})">${t}</div>`
    ).join('');
  }
  
  function renderMatrix() {
    const matrixEl = document.getElementById('attn-matrix');
    if (!matrixEl) return;
    matrixEl.style.gridTemplateColumns = `repeat(${TOKENS.length}, 1fr)`;
    
    const weights = ATTN[selected];
    matrixEl.innerHTML = weights.map((w, i) => {
      const color = getColor(w);
      const textColor = w > 0.3 ? '#1a1a1a' : '#666';
      return `<div class="attn-cell" style="background:${color};color:${textColor}">${w > 0 ? w.toFixed(2) : '-'}</div>`;
    }).join('');
  }
  
  function renderInfo() {
    const infoEl = document.getElementById('attn-info');
    if (!infoEl) return;
    const token = TOKENS[selected];
    const weights = ATTN[selected];
    const topIdx = weights.indexOf(Math.max(...weights));
    const topToken = TOKENS[topIdx];
    const topWeight = weights[topIdx];
    
    infoEl.innerHTML = `
      <strong>Token "${token}" (position ${selected})</strong><br>
      <div class="formula">attention = softmax(Q·K<sup>T</sup> / √d<sub>k</sub>) · V</div>
      Most attended token: <strong>"${topToken}"</strong> (weight: ${topWeight.toFixed(2)})<br>
      <br>
      <em>Causal masking:</em> Can only attend to positions 0..${selected} (not future tokens)
    `;
  }
  
  window.attnSelect = function(idx) {
    selected = idx;
    renderTokens();
    renderMatrix();
    renderInfo();
  };
  
  renderTokens();
  renderMatrix();
  renderInfo();
})();
</script>

## Part 3: GPT Architecture Overview

### Full Architecture

```
Input: Token IDs [5, 12, 3, 8, ...]
  ↓
Token Embedding (vocab_size → d_model)
  + Positional Encoding (tells model position of each token)
  ↓
Transformer Block 1:
  - Multi-Head Self-Attention (4 heads)
  - Layer Normalization
  - Residual connection
  - Feed-Forward Network (expand → compress)
  - Layer Normalization
  - Residual connection
  ↓
Transformer Block 2:
  ...(same as Block 1)
  ↓
Transformer Block 3:
  ...
  ↓
Transformer Block 4:
  ...
  ↓
Final Layer Norm
  ↓
Linear (d_model → vocab_size)
  ↓
Output: Logits for each token
```

### Key Components

**1. Token Embedding:** Convert token IDs to continuous vectors
- Input: (batch, seq_len)
- Output: (batch, seq_len, d_model)

**2. Positional Encoding:** Add position information
- Without this, model can't distinguish position
- "cat sat" vs "sat cat" would look identical
- Two approaches: learned embeddings or sinusoidal

**3. Multi-Head Attention:** Attend to different aspects
- Head 1: Might learn subject-verb relationships
- Head 2: Might learn noun-adjective relationships
- Head 3: Might learn long-range dependencies
- Head 4: Might learn local patterns
- Combine all heads for rich representation

**4. Feed-Forward Network:** Process attended information
- Two linear layers with ReLU
- Expand then compress: d_model → 4*d_model → d_model
- Applied to each position independently

**5. Layer Normalization:** Stabilize training
- Normalize across embedding dimension
- Prevents exploding/vanishing gradients

**6. Residual Connections:** Enable deep networks
- x = x + Attention(x)
- x = x + FFN(x)
- Gradients flow directly through
- Can stack 100+ layers

**7. Causal Masking:** Can't see future tokens
- When predicting token i, can only see tokens 0..i-1
- Ensures autoregressive generation

### Hyperparameters for This Lab

- `d_model = 128`: Embedding dimension
- `n_heads = 4`: Number of attention heads
- `n_layers = 4`: Number of transformer blocks
- `d_ff = 512`: Feed-forward hidden dimension
- `block_size = 64`: Maximum context length
- `dropout = 0.1`: Regularization

**Result:** ~1.2M parameters (vs makemore's ~30K)

## Exercise 1: Implement NanoGPT Model

### Your Task

Build a GPT model using PyTorch's `nn.TransformerDecoderLayer`:
1. Token embedding layer
2. Positional embedding (learned)
3. Transformer decoder layers (use PyTorch's built-in)
4. Final layer norm and output projection

### Hints
- Use `nn.Embedding` for both token and position embeddings
- Use `nn.TransformerDecoderLayer` for attention + FFN
- Use `nn.TransformerDecoder` to stack layers
- Use `nn.Transformer.generate_square_subsequent_mask()` for causal masking
- Set `batch_first=True` in decoder layer

In [ ]:
class NanoGPT(nn.Module):
    """Transformer decoder for language modeling."""
    
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=4, d_ff=512, 
                 block_size=64, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.block_size = block_size
        
        # Token embedding
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        
        # Positional embedding (learned)
        self.pos_embedding = nn.Embedding(block_size, d_model)
        
        # Transformer decoder layers
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, n_layers)
        
        # Final layer norm and output projection
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx):
        """
        Args:
            idx: (B, T) tensor of token IDs
        Returns:
            logits: (B, T, vocab_size)
        """
        B, T = idx.shape
        
        # Get token embeddings
        tok_emb = self.token_embedding(idx)  # (B, T, d_model)
        
        # Get positional embeddings
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        pos_emb = self.pos_embedding(pos)  # (T, d_model)
        
        # Combine (broadcasting)
        x = tok_emb + pos_emb  # (B, T, d_model)
        
        # Create causal mask
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=idx.device)
        
        # Apply transformer
        x = self.transformer(x, x, tgt_mask=causal_mask, memory_mask=causal_mask)
        
        # Final layer norm and projection
        x = self.ln_f(x)
        logits = self.head(x)  # (B, T, vocab_size)
        
        return logits

# Create model
model = NanoGPT(
    vocab_size=tokenizer.vocab_size,
    d_model=128,
    n_heads=4,
    n_layers=4,
    d_ff=512,
    block_size=64
)
model = model.to(device)
print(f"NanoGPT: {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"\n✓ NanoGPT model ready!")

## Exercise 2: Implement Training Loop

### Training Setup

**Differences from Makemore:**
- Longer context: block_size=64 (vs 8 for makemore)
- Gradient clipping: Prevent exploding gradients in deep network
- Predict all positions: Not just last token

**Expected Results:**
- Loss should decrease from ~4.2 to ~1.6
- Perplexity should decrease from ~65 to ~5.0
- Training takes ~8 minutes on T4 GPU
- Better than makemore's perplexity of ~7.5!

### Your Task

Train the NanoGPT model:
1. Get batch with longer context (block_size=64)
2. Forward pass: compute logits for all positions
3. Compute cross-entropy loss (flatten for all positions)
4. Backward pass with gradient clipping
5. Track and print metrics

In [ ]:
# Training settings
batch_size = 64
block_size = 64
max_iters = 3000
eval_interval = 300
learning_rate = 3e-4
grad_clip = 1.0

# Create model
model = NanoGPT(
    vocab_size=tokenizer.vocab_size,
    d_model=128,
    n_heads=4,
    n_layers=4,
    d_ff=512,
    block_size=block_size
)
model = model.to(device)

# Create optimizer
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

# Training loop
losses = []

print("Training NanoGPT (this will take ~8 minutes on GPU)...\n")

for iter in tqdm(range(max_iters), desc="Training"):
    # Get batch
    xb, yb = get_batch('train', train_data, val_data, batch_size, block_size, device)
    
    # Forward - predict all positions
    logits = model(xb)  # (B, T, vocab_size)
    B, T, C = logits.shape
    logits = logits.view(B*T, C)
    targets = yb.view(B*T)
    loss = F.cross_entropy(logits, targets)
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    
    # Gradient clipping
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    
    optimizer.step()
    
    losses.append(loss.item())
    
    # Evaluate
    if iter % eval_interval == 0 or iter == max_iters - 1:
        model.eval()
        with torch.no_grad():
            xb_val, yb_val = get_batch('val', train_data, val_data, batch_size, block_size, device)
            logits_val = model(xb_val)
            B, T, C = logits_val.shape
            logits_val = logits_val.view(B*T, C)
            targets_val = yb_val.view(B*T)
            val_loss = F.cross_entropy(logits_val, targets_val)
        model.train()
        
        perplexity = torch.exp(val_loss).item()
        print(f"\nIter {iter}: train loss {loss.item():.4f}, val loss {val_loss.item():.4f}, perplexity {perplexity:.2f}")

print("\n✓ Training complete!")

### Plot Training Loss

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('NanoGPT Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

## Part 4: Sampling Strategies

### Why Sampling Matters

When generating text, how do we choose the next token?

### Temperature Sampling

**Formula:** `probs = softmax(logits / temperature)`

**Effect:**
- `temperature < 1.0`: More confident (sharper distribution)
  - temperature=0.1: Almost deterministic
  - Good for factual tasks
- `temperature = 1.0`: Normal sampling
- `temperature > 1.0`: More random (flatter distribution)
  - temperature=2.0: Very creative but risky
  - Good for creative tasks

**Example:**
```
Logits: [3.0, 2.0, 1.0, 0.5]

Temperature = 0.5 (confident):
probs = [0.70, 0.22, 0.06, 0.02]
→ Almost always pick first token

Temperature = 1.0 (normal):
probs = [0.49, 0.27, 0.15, 0.09]
→ Balanced sampling

Temperature = 2.0 (creative):
probs = [0.38, 0.28, 0.20, 0.14]
→ Much more diverse
```

### Top-K Sampling

**Idea:** Only sample from top k most likely tokens

**Benefits:**
- Prevents sampling very unlikely tokens
- More coherent than pure random sampling
- Still allows diversity

**Example (k=3):**
```
All tokens: ["the": 0.5, "a": 0.2, "an": 0.15, "some": 0.1, "xyz": 0.05]
Top-3: ["the": 0.5, "a": 0.2, "an": 0.15]
Renormalize: ["the": 0.59, "a": 0.24, "an": 0.18]
Sample only from these 3
```

### Combining Strategies

In practice, combine temperature + top-k:
1. Apply temperature to logits
2. Filter to top-k tokens
3. Renormalize and sample

**Result:** Coherent yet diverse text

## Exercise 3: Implement Generation with Sampling

### Your Task

Implement text generation with temperature and top-k sampling:
1. Start with prompt (encoded to token IDs)
2. For each new token:
   - Get model predictions
   - Apply temperature
   - Apply top-k filtering
   - Sample from distribution
   - Append to sequence
3. Decode and return generated text

In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=200, temperature=1.0, top_k=None, device='cpu'):
    """
    Generate text from model with temperature and top-k sampling.
    
    Args:
        model: trained model
        tokenizer: tokenizer
        prompt: starting text string
        max_new_tokens: number of tokens to generate
        temperature: sampling temperature (higher = more random)
        top_k: if set, only sample from top k tokens
        device: device to run on
    Returns:
        generated text
    """
    model.eval()
    
    # Encode prompt
    ids = tokenizer.encode(prompt)
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    
    # Generate tokens
    for _ in range(max_new_tokens):
        # Crop to block_size
        x_cond = x if x.size(1) <= model.block_size else x[:, -model.block_size:]
        
        # Get predictions
        with torch.no_grad():
            logits = model(x_cond)  # (1, T, vocab_size)
            logits = logits[:, -1, :]  # (1, vocab_size)
        
        # Apply temperature
        logits = logits / temperature
        
        # Top-k filtering
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')
        
        # Sample
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        # Append
        x = torch.cat([x, next_token], dim=1)
    
    # Decode
    generated_ids = x[0].tolist()
    return tokenizer.decode(generated_ids)

# Test generation
model.eval()
prompt = "To be or not to be"
print(f"Prompt: '{prompt}'\n")
print("="*80)
generated = generate(model, tokenizer, prompt, max_new_tokens=200, temperature=1.0, device=device)
print(generated)
print("="*80)

### Compare Different Temperatures

In [ ]:
model.eval()
prompt = "ROMEO:"

print(f"Prompt: '{prompt}'\n")
print("="*80)

for temp in [0.5, 0.8, 1.0, 1.2]:
    print(f"\nTemperature = {temp}:")
    print("-"*80)
    generated = generate(model, tokenizer, prompt, max_new_tokens=150, temperature=temp, device=device)
    print(generated)

print("\n" + "="*80)
print("\nObservations:")
print("- Low temp (0.5): More conservative, repetitive, stays on topic")
print("- Medium temp (0.8-1.0): Balanced, good quality")
print("- High temp (1.2+): More creative but potentially incoherent")

### Compare Top-K Sampling

In [ ]:
prompt = "JULIET:"

print(f"Prompt: '{prompt}'\n")
print("="*80)

for k in [5, 10, 20, 40]:
    print(f"\nTop-k = {k}:")
    print("-"*80)
    generated = generate(model, tokenizer, prompt, max_new_tokens=150, temperature=0.8, top_k=k, device=device)
    print(generated)

print("\n" + "="*80)
print("\nObservations:")
print("- Small k (5): Very focused, less diverse")
print("- Medium k (10-20): Good balance")
print("- Large k (40+): More diverse, closer to pure sampling")

## Part 5: Comparison with Makemore

Let's compare the quality of text generated by NanoGPT vs Makemore

In [ ]:
# Compare with makemore
print("Comparing NanoGPT (Transformer) vs Makemore (MLP)")
print("="*80)
print("\nPrompt: 'KING:'\n")

# NanoGPT
print("NanoGPT (64-token context, 4-layer transformer, perplexity ~5.0):")
print("-"*80)
generated_gpt = generate(model, tokenizer, "KING:", max_new_tokens=200, temperature=0.8, top_k=10, device=device)
print(generated_gpt)

print("\n" + "="*80)
print("\nKey Improvements:")
print("✓ NanoGPT: Better perplexity (5.0 vs 7.5)")
print("✓ NanoGPT: Longer context (64 vs 8 characters)")
print("✓ NanoGPT: More coherent sentences")
print("✓ NanoGPT: Better long-range dependencies")
print("✓ NanoGPT: Can attend to relevant past tokens")
print("\n✗ Makemore: Limited to 8-char context")
print("✗ Makemore: No attention mechanism")
print("✗ Makemore: Treats all context equally")

## Part 6: Save Checkpoint

Save the trained model for later use

In [ ]:
import os

os.makedirs('checkpoints', exist_ok=True)

checkpoint = {
    'model_state_dict': model.state_dict(),
    'vocab_size': tokenizer.vocab_size,
    'd_model': 128,
    'n_heads': 4,
    'n_layers': 4,
    'd_ff': 512,
    'block_size': 64,
    'val_loss': val_loss.item(),
}

torch.save(checkpoint, 'checkpoints/nanogpt_model.pt')
print("✓ Checkpoint saved to checkpoints/nanogpt_model.pt")
print(f"\nFinal validation perplexity: {torch.exp(val_loss).item():.2f}")

## Part 7: What's Next?

### From NanoGPT to GPT-3

What we built:
- Character-level tokenization
- 128 embedding dimension
- 4 layers, 4 attention heads
- ~1.2M parameters
- Trained on 1MB of text
- Context: 64 tokens

**GPT-2:**
- Subword (BPE) tokenization
- 1024-1600 embedding dimension
- 12-48 layers, 12-25 attention heads
- 117M - 1.5B parameters
- Trained on 40GB of text
- Context: 1024 tokens

**GPT-3:**
- Subword (BPE) tokenization
- 12,288 embedding dimension
- 96 layers, 96 attention heads
- 175 billion parameters
- Trained on ~500B tokens (~5TB of text)
- Context: 2048 tokens (GPT-4: 128K tokens!)

**Same architecture, just scaled up!**

### Subword Tokenization (BPE)

**Why not characters?**
- Character sequences are very long
- Hard to learn word meanings from individual chars
- Computationally expensive for long documents

**Byte-Pair Encoding (BPE):**
- Start with characters
- Merge most frequent pairs
- Repeat until vocab size reached (e.g., 50k)
- Result: Common words = 1 token, rare words = multiple tokens

**Example:**
```
Text: "unhappiness"
Char-level: ['u','n','h','a','p','p','i','n','e','s','s'] (11 tokens)
BPE: ['un','happy','ness'] (3 tokens)
```

### Key Improvements Beyond This Lab

**1. Flash Attention:** Faster attention computation
- Standard attention: O(n²) memory
- Flash attention: O(n) memory
- Enables much longer context

**2. Mixture of Experts (MoE):** Scale without proportional compute
- Different experts for different inputs
- Only activate relevant experts
- Used in GPT-4, Gemini

**3. RLHF (Reinforcement Learning from Human Feedback):**
- Fine-tune with human preferences
- Makes model helpful, honest, harmless
- Used in ChatGPT, Claude

**4. Multimodal:**
- Process text + images + audio
- Unified architecture for all modalities
- GPT-4V, Claude 3

### Try These Next

**1. Fine-tune GPT-2 on custom dataset**
- Use Hugging Face `transformers` library
- Load pre-trained GPT-2
- Fine-tune on your domain (code, poetry, etc.)

**2. Build a chatbot**
- Add conversation history as context
- Implement turn-taking
- Add system prompts

**3. Implement different architectures**
- BERT (encoder-only, for classification)
- T5 (encoder-decoder, for seq2seq)
- Vision Transformer (for images)

**4. Explore quantization and optimization**
- 8-bit and 4-bit quantization
- LoRA (Low-Rank Adaptation) for efficient fine-tuning
- Distillation to smaller models

## Summary

### What You've Learned

✅ **Why transformers:** Attention + longer context = better performance

✅ **Attention mechanism:** Query, Key, Value - learn what to focus on

✅ **Transformer architecture:** Embedding + attention + FFN + normalization

✅ **Causal masking:** Prevent looking at future tokens

✅ **Multi-head attention:** Attend to different aspects simultaneously

✅ **Training:** Gradient clipping, longer context, predict all positions

✅ **Sampling strategies:** Temperature and top-k for generation

✅ **Performance:** NanoGPT achieves perplexity ~5.0 vs makemore's ~7.5

### Complete Lab 6 Journey

**Part 1: Makemore**
- Character tokenization
- MLP language model
- Fixed 8-char context
- Perplexity: ~7.5

**Part 2: NanoGPT**
- Transformer architecture
- Attention mechanism
- 64-token context
- Perplexity: ~5.0

### From Scratch to State-of-the-Art

**Lab 1:** Pure Python → Understood arrays

**Lab 2:** Autograd → Understood gradients

**Lab 3:** Neural networks → Understood MLPs

**Lab 4:** PyTorch → Understood frameworks

**Lab 5:** Transfer learning → Understood modern deep learning

**Lab 6:** Transformers → Understood language models

**You now understand the fundamentals behind GPT, BERT, and modern LLMs!**

### Congratulations!

You've completed the micrograd-lab series. You now have a deep understanding of:
- How neural networks work from first principles
- How automatic differentiation works
- How transformers and attention work
- How to train and deploy models

**Next steps:**
- Explore Hugging Face Transformers library
- Fine-tune pre-trained models
- Build production ML systems
- Contribute to open-source ML projects

**Happy learning and building!**